# 🇲🇲 AI Voice Studio — VoxCPM2 Myanmar Voice Clone
**Phone-only test:** open this notebook in Google Colab from your phone. Select a GPU runtime, run the cells from top to bottom, upload a reference voice, enter Myanmar text, and download the WAV.

Model: `openbmb/VoxCPM2` • Burmese/Myanmar supported • 2B parameters • about 4.96 GB model files.

This notebook intentionally uses direct notebook controls rather than a persistent public Gradio/web server. Google states that free Colab resources are not guaranteed or unlimited and that managed free runtimes may terminate content-generation web-UI workloads.

In [ ]:
!pip -q install -U voxcpm librosa soundfile
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is required for this practical test. In Colab choose Runtime > Change runtime type > GPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))

In [ ]:
from voxcpm import VoxCPM
import numpy as np
import librosa
import soundfile as sf
from google.colab import files
import os

MODEL_ID = 'openbmb/VoxCPM2'
model = VoxCPM.from_pretrained(MODEL_ID, load_denoiser=False)
SR = int(getattr(model.tts_model, 'sample_rate', 48000))
print('VoxCPM2 ready. Sample rate:', SR)

In [ ]:
# 1) Upload a short reference voice recording (WAV/MP3/M4A supported by VoxCPM's audio loader).
uploaded = files.upload()
reference_audio = next(iter(uploaded))
print('Reference:', reference_audio)

In [ ]:
# 2) Enter Myanmar text and speed. Keep the text <= 5,000 characters.
text = '''မင်္ဂလာပါ။ ဒီအသံကို ကျွန်တော်တို့ AI Voice Studio နဲ့ စမ်းသပ်ထုတ်လုပ်နေပါတယ်။'''
speed = 1.0  # 0.5 to 2.0

def has_myanmar(text):
    return any('\u1000' <= c <= '\u109f' or '\uaa60' <= c <= '\uaa7f' for c in text)

text = (text or '').strip()
if not text:
    raise ValueError('Myanmar text is empty.')
if len(text) > 5000:
    raise ValueError('Maximum 5,000 characters.')
if not has_myanmar(text):
    raise ValueError('Please enter Myanmar/Burmese text.')
speed = max(0.5, min(2.0, float(speed)))
print('Characters:', len(text), '| Speed:', speed)

In [ ]:
# 3) Generate the cloned Myanmar voice.
wav = model.generate(
    text=text,
    reference_wav_path=reference_audio,
    cfg_value=2.0,
    inference_timesteps=10,
)
audio = wav.detach().float().cpu().numpy() if hasattr(wav, 'detach') else np.asarray(wav)
audio = np.squeeze(audio)
if abs(speed - 1.0) > 0.001:
    audio = librosa.effects.time_stretch(audio, rate=speed)
output_path = '/content/myanmar-voice-clone.wav'
sf.write(output_path, audio, SR)
print('DONE:', output_path, '| Sample rate:', SR)
files.download(output_path)